# Analyzing Open Problems results in a smaller scope

- Using the already existing results of Open Problems (ran between 2024.09.23-2025.10.11)
- 10 Datasets
    - mouse pancreas atlas
    - diabetic kidney disease
    - immune cell atlas
    - gtex_vg
    - hypomap
    - tabula sapiens
    - zebrafish
    - cengen
    - human immune (immune_cells: from scib benchmark)
    - human pancreas (pancreas: from scib benchmark)
    
  including following dataset features:
    - organism
    - n_tissues
    - size
    - n_genes
    - n_celltypes
    - n_batches
    - batch_imbalance (Coeffiecient of Variance)
    - nested_batch
    - n_donors
    - n_samples
- The metric `kBET` will be excluded since it fails when running on mouse pancreas atlas
- Methods that fail often or consistantly perform poorly are excluded (below (and including) BBKNN)

## setting up the packages and dataframe

In [3]:
import os
import yaml
import pandas as pd
import matplotlib.pyplot as plt
import time
import seaborn as sns  # optional, just for color if you like
import matplotlib.pyplot as plt

In [8]:
base_dir = "/Users/seohyon/resources/results"

# will track only these methods (above BBKNN)
keep_methods = [
    "embed_cell_types_jittered",
    "embed_cell_types",
    "shuffle_integration",
    "scanvi",
    "combat",
    "scvi",
    "harmony",
    "harmonypy",
    "batchelor_fastmnn",
    "liger",
    "pyliger",
    "scalex",
    "uce",
    "no_integration",
    "no_integration_batch",
]

rows = []

# Step 1 — Collect all rows
for folder in os.listdir(base_dir):
    folder_path = os.path.join(base_dir, folder)
    yaml_path = os.path.join(folder_path, "score_uns.yaml")
    
    if os.path.isdir(folder_path) and os.path.exists(yaml_path):
        with open(yaml_path, "r") as f:
            try:
                data = yaml.safe_load(f)
            except yaml.YAMLError as e:
                print(f"⚠️ Could not read {yaml_path}: {e}")
                continue

        for entry in data:
            method = entry.get("method_id", "").strip().lower()

            # ✅ keep only methods in your whitelist
            if method not in keep_methods:
                continue

            metrics = entry.get("metric_ids", [])
            values = entry.get("metric_values", [])

            for m_id, m_val in zip(metrics, values):
                rows.append({
                    "dataset_id": entry.get("dataset_id"),
                    # "file_size": entry.get("file_size"),
                    "method_id": method,
                    # "normalization_id": entry.get("normalization_id"),
                    "metric_id": m_id,
                    "metric_value": m_val
                })

# Step 2 — Create one big DataFrame
df = pd.DataFrame(rows)
df = df.dropna(subset=["metric_value"]).copy() # drop NaN
df = df[df["metric_id"] != "kbet"].copy() # drop kbet
df["dataset_id"] = df["dataset_id"].str.split("/").str[-1] # clean the dataset id

# # Step 3 — Split into separate tables per dataset_id
# datasets = {}
# for dataset_id, sub_df in df.groupby("dataset_id"):
#     datasets[dataset_id] = sub_df.reset_index(drop=True)
#     print(f"\n=== Dataset: {dataset_id} ===")
#     print(sub_df)

In [9]:
df["dataset_id"].unique()

array(['zebrafish', 'dkd', 'tabula_sapiens', 'pancreas',
       'mouse_pancreas_atlas', 'cengen', 'gtex_v9', 'hypomap',
       'immune_cells', 'immune_cell_atlas'], dtype=object)